# Enrich Court Authority Cards — Qwen3.5-35B-A3B with vLLM

Enriches `court_authority_cards_v4.jsonl` with LLM-derived semantic fields for RAG.

**Model:** `Qwen/Qwen3.5-35B-A3B`  
**Runtime:** vLLM offline inference (`LLM.generate`)  
**GPU target:** single high-VRAM NVIDIA GPU, e.g. RTX PRO 6000 Blackwell / G4-class 96 GB VRAM  
**Structured output:** vLLM `StructuredOutputsParams(json=RAG_SCHEMA)` when available, with a compatibility fallback for older vLLM builds.  
**Thinking mode:** disabled at chat-template rendering time with `enable_thinking=False`.

The notebook is vLLM-only. It does not use Hugging Face `model.generate()`, `torch.no_grad()`, or manual tokenization for inference.


## 1 · Environment setup

In [1]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    # Verify GPU
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip())

Running in Colab: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [2]:
# Fresh runtime strongly recommended before running this cell.
# Do not import vllm / transformers / torch / torchvision / PIL before this cell.

%%bash
set -e

echo "Python executable:"
which python
python --version

echo "Cleaning broken / mixed packages..."
python -m pip uninstall -y -q PIL pillow numpy scipy torchvision vllm transformers || true

echo "Installing uv + basic utilities..."
python -m pip install -q -U uv tqdm

echo "Writing constraints..."
cat > /tmp/vllm_constraints.txt <<'EOF'
pillow==11.3.0
numpy==2.3.5
scipy==1.16.3
EOF

echo "Installing pinned numeric/image stack first..."
python -m pip install -q --no-cache-dir --force-reinstall \
  -c /tmp/vllm_constraints.txt \
  pillow numpy scipy

echo "Installing vLLM with official torch backend resolver..."
uv pip install --system \
  vllm \
  --torch-backend=auto \
  -c /tmp/vllm_constraints.txt

echo "Running dependency sanity check..."
python -m pip check || true

echo "Running import sanity check..."
python - <<'PY'
import sys
print("Python:", sys.version)
print("Executable:", sys.executable)

import PIL
from PIL import Image, ImageDraw
print("Pillow OK:", PIL.__version__, PIL.__file__)

import numpy as np
import scipy
print("NumPy OK:", np.__version__, np.__file__)
print("SciPy OK:", scipy.__version__, scipy.__file__)

# This was the previous failing NumPy internal import path.
from numpy._core.umath import _center
print("NumPy internal symbol OK")

import torch
print("Torch OK:", torch.__version__, "CUDA:", torch.version.cuda, "available:", torch.cuda.is_available())

try:
    import torchvision
    print("Torchvision OK:", torchvision.__version__)
except Exception as e:
    print("Torchvision import warning:", repr(e))

from vllm import LLM, SamplingParams
print("vLLM import OK")

try:
    import vllm
    print("vLLM version:", getattr(vllm, "__version__", "unknown"))
except Exception:
    pass
PY

echo "Install/import cell completed."

Python executable:
/usr/local/bin/python
Python 3.12.13
Cleaning broken / mixed packages...
Installing uv + basic utilities...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 115.7 MB/s eta 0:00:00
Writing constraints...
Installing pinned numeric/image stack first...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 20.2 MB/s eta 0:00:00
Installing vLLM with official torch backend resolver...
Running dependency sanity check...
ipython 7.34.0 requires jedi, which is not installed.
libraft-cu12 26.2.0 has requirement cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2.
pylibraft-cu12 26.2.0 has requirement cuda-python<13.0,>=12

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
peft 0.19.1 requires transformers, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
Using Python 3.12.13 environment at: /usr
Resolved 180 packages in 2.86s
Prepared 77 packages in 15.81s
Uninstalled 9 packages in 907ms
Installed 77 packages in 266ms
 + anthropic==0.97.0
 + apache-tvm-ffi==0.1.9
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==6.0.1
 + compressed-tensors==0.15.0.1
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-python==12.9.4
 + cuda-python==13.2.0
 + cuda-tile==1.3.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email

In [2]:
import sys, subprocess, site, os

print("Python:", sys.version)
print("Executable:", sys.executable)

subprocess.run([sys.executable, "-m", "pip", "show", "pillow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "Pillow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "torch"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "torchvision"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "vllm"], check=False)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

import PIL
print("PIL file:", PIL.__file__)
print("PIL version:", PIL.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
PIL file: /usr/local/lib/python3.12/dist-packages/PIL/__init__.py
PIL version: 11.3.0


In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Configuration

In [4]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR    = BASE_DIR / 'data'
INSIGHTS_DIR= BASE_DIR / 'data_insights'
ART_DIR     = BASE_DIR / 'artifacts'
SCRIPT_DIR  = BASE_DIR / 'scripts'

for d in [DATA_DIR, INSIGHTS_DIR, ART_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────────────────
# Official Qwen3.5-35B-A3B chat / multimodal checkpoint.
# The HF repo is ~71.9 GB of safetensors. On a ~96 GB GPU, keep context and
# batch size conservative for the first run, then increase after a clean load.
MODEL_ID     = 'Qwen/Qwen3.5-35B-A3B'
QUANTIZATION = None     # None for BF16. Use a vLLM-supported quantized checkpoint separately.

# ── Inference ────────────────────────────────────────────────────────────────
GPU_MEMORY_UTIL    = 0.90   # Start conservative on 95.8 GB VRAM; raise to 0.92 only if stable.
MAX_MODEL_LEN      = 4096
BATCH_SIZE         = 4      # Safe first-run value. Try 8/16 after confirming no OOM.
TEMPERATURE        = 0.15   # Lower = more deterministic JSON.
MAX_TOKENS         = 600
TENSOR_PARALLEL    = 1      # Single GPU; set 2+ only on a multi-GPU runtime.

# vLLM recipe: latency-focused mode uses MTP-1 speculative decoding and disables prefix caching.
USE_MTP            = True
MTP_TOKENS         = 1
ENABLE_PREFIX_CACHING = False

# Text-only workload. If the installed vLLM exposes the Python equivalent of
# --language-model-only, the loader cell will enable it automatically.
LANGUAGE_MODEL_ONLY = True

# ── Files ────────────────────────────────────────────────────────────────────
INPUT_FILE      = ART_DIR / 'court_authority_cards_v4.jsonl'
OUTPUT_FILE     = ART_DIR / 'court_authority_cards_rag.jsonl'
CHECKPOINT_FILE = ART_DIR / 'rag_checkpoint.txt'

# Optional: stop after N cards (0 = process all)
LIMIT = 1000

print('BASE_DIR   :', BASE_DIR)
print('INPUT_FILE :', INPUT_FILE)
print('OUTPUT_FILE:', OUTPUT_FILE)
print('MODEL_ID   :', MODEL_ID)


BASE_DIR   : /content/drive/MyDrive/swiss_law
INPUT_FILE : /content/drive/MyDrive/swiss_law/artifacts/court_authority_cards_v4.jsonl
OUTPUT_FILE: /content/drive/MyDrive/swiss_law/artifacts/court_authority_cards_rag.jsonl
MODEL_ID   : Qwen/Qwen3.5-35B-A3B


## 3 · JSON schema + prompts

In [5]:
import re

# ── Pre-filter: trivial paragraphs that don't need an LLM ────────────────────
COST_PROC_RE = re.compile(
    r'(?:'
    r'\bgerichtskosten\b|\bprozesskosten\b|\bverfahrenskosten\b|'
    r'\bfrais judiciaires\b|\bfrais de la cause\b|\bd[eé]pens\b|'
    r'\bspese giudiziarie\b|\bripetibili\b|'
    r'\bparteientsch[äa]digung\b|\bhonoraire\b|'
    r'\bunentgeltliche rechtspflege\b|\bassistance judiciaire\b|'
    r'\bpatrocinio gratuito\b|'
    r'\bdie sache wird .{0,80}zur[üu]ckgewiesen\b|'
    r'\brenvoyer la cause\b|'
    r'\bla causa [eè] rinviata\b|'
    r'^\s*\d+\.\s*\d+\..{0,5}fr\.\s*\d'
    r')',
    re.IGNORECASE | re.MULTILINE,
)

# ── JSON schema for guided generation ────────────────────────────────────────
RAG_SCHEMA = {
    'type': 'object',
    'properties': {
        'english_summary':          {'type': 'string'},
        'legal_topic':              {'type': 'string'},
        'legal_question':           {'type': 'string'},
        'legal_rule':               {'type': 'string'},
        'court_holding':            {'type': 'string'},
        'factual_context':          {'type': 'string'},
        'english_legal_concepts':   {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
        'search_keywords':          {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 10},
        'natural_language_queries': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
        'paragraph_role': {
            'type': 'string',
            'enum': ['holding', 'reasoning', 'background', 'cost',
                     'procedural', 'disposition', 'standard_of_review', 'obiter'],
        },
        'outcome_signal': {
            'type': 'string',
            'enum': ['granted', 'dismissed', 'inadmissible', 'remitted', 'partial', 'none'],
        },
    },
    'required': [
        'english_summary', 'legal_topic',
        'english_legal_concepts', 'search_keywords',
        'natural_language_queries', 'paragraph_role', 'outcome_signal',
    ],
    'additionalProperties': False,
}

SYSTEM_PROMPT = (
    'You are a Swiss legal analyst. The user gives you a paragraph from a Swiss '
    'Federal Tribunal decision in German, French, or Italian. '
    'Translate every concept into precise English legal terminology and emit '
    'structured JSON to power English-language semantic-search RAG. '
    'Be concrete: prefer \'extension of pretrial detention based on flight risk\' '
    'over \'detention\'. Output ONLY the JSON object, no preamble.'
)

print('Schema fields:', list(RAG_SCHEMA['properties'].keys()))


Schema fields: ['english_summary', 'legal_topic', 'legal_question', 'legal_rule', 'court_holding', 'factual_context', 'english_legal_concepts', 'search_keywords', 'natural_language_queries', 'paragraph_role', 'outcome_signal']


## 4 · Helper functions

In [6]:
import json
from typing import Iterator


def build_user_message(card: dict) -> str:
    text       = card.get('text_excerpt_original', '')[:2500]
    citation   = card.get('citation', '')
    legal_area = card.get('legal_area', '')
    existing   = card.get('issue_labels_en') or []
    parts = [
        f'Citation: {citation}',
        f'Legal area (deterministic): {legal_area}',
    ]
    if existing:
        labels = ', '.join(existing[:8])
        parts.append(f'Existing labels: {labels}')
    parts += ['', 'Paragraph (original language):', text, '', 'Produce the JSON now.']
    return '\n'.join(parts)


def _stub(summary, topic, concepts, keywords, role, method):
    return {
        'english_summary':          summary,
        'legal_topic':              topic,
        'legal_question':           '',
        'legal_rule':               '',
        'court_holding':            '',
        'factual_context':          '',
        'english_legal_concepts':   concepts,
        'search_keywords':          keywords,
        'natural_language_queries': [],
        'paragraph_role':           role,
        'outcome_signal':           'none',
        'method':                   method,
    }


def auto_classify(card: dict) -> dict | None:
    if card.get('is_notification_paragraph'):
        return _stub(
            'Procedural notification of the judgment to the parties.',
            'judgment notification',
            ['service of judgment'],
            ['notification', 'service', 'judgment communication'],
            role='procedural', method='auto_notification',
        )
    text = card.get('text_excerpt_original', '') or ''
    if len(text) < 50:
        return _stub(
            'Short procedural fragment (cross-reference or one-line ruling).',
            'procedural fragment',
            [], [], role='procedural', method='auto_short',
        )
    if COST_PROC_RE.search(text[:400]):
        return _stub(
            'Court-cost or procedural-fee allocation paragraph.',
            'court costs and procedural fees',
            ['court costs', 'procedural fees', 'legal aid'],
            ['costs', 'court fees', 'frais judiciaires', 'Gerichtskosten'],
            role='cost', method='auto_cost',
        )
    return None


def stream_input(path: Path, start_offset: int) -> Iterator[tuple[int, dict]]:
    with path.open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < start_offset or not line.strip():
                continue
            try:
                yield i, json.loads(line)
            except json.JSONDecodeError:
                continue


def count_lines(path: Path) -> int:
    n = 0
    with path.open('rb') as f:
        for _ in f:
            n += 1
    return n


print('Helpers loaded.')

Helpers loaded.


## 5 · Load model with vLLM

The notebook follows the vLLM Qwen3.5 recipe:

- Uses vLLM offline inference (`LLM` + `llm.generate`) instead of Transformers `model.generate`.
- Uses Qwen's `reasoning_parser='qwen3'` when the installed vLLM build accepts that engine argument.
- Uses latency-focused settings from the recipe: MTP-1 speculative decoding and disabled prefix caching.
- Uses `StructuredOutputsParams(json=RAG_SCHEMA)` on current vLLM builds, with a fallback to `GuidedDecodingParams` on older builds.
- Disables thinking at prompt-rendering time with `enable_thinking=False`.


In [7]:
import vllm
print('vLLM version:', getattr(vllm, '__version__', 'unknown'))


vLLM version: 0.20.0


In [8]:
# Optional runtime sanity check.
if IN_COLAB:
    !nvidia-smi


Thu Apr 30 18:59:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
# No Transformers model/tokenizer install is needed for inference.
# vLLM owns model loading, scheduling, KV cache, batching, and generation.


In [10]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"



In [11]:
from vllm import LLM, SamplingParams

In [12]:
os.environ.pop("VLLM_MOE_BACKEND", None)
os.environ.pop("VLLM_FLASHINFER_MOE_BACKEND", None)

In [13]:
import os
import inspect

# Must be set before importing/initializing vLLM.
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# Remove stale/invalid env vars from earlier attempts.
os.environ.pop("VLLM_WORKER_MULTIPROC_METHOD", None)
os.environ.pop("VLLM_MOE_BACKEND", None)
os.environ.pop("VLLM_FLASHINFER_MOE_BACKEND", None)

from vllm import LLM, SamplingParams
from vllm.config import KernelConfig

try:
    from vllm.sampling_params import StructuredOutputsParams
except Exception:
    StructuredOutputsParams = None

try:
    from vllm.sampling_params import GuidedDecodingParams
except Exception:
    GuidedDecodingParams = None


MODEL_ID = "Qwen/Qwen3.5-35B-A3B"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTIL = 0.90

# Aggressive throughput settings.
BATCH_SIZE = 32

# Safer aggressive default. If outputs are consistently short, change to 256.
MAX_TOKENS = 384
# MAX_TOKENS = 256

TEMPERATURE = 0.15

print("Loading Qwen3.5 with vLLM offline inference...")
print(f"MODEL_ID={MODEL_ID}")
print(
    f"max_model_len={MAX_MODEL_LEN}, "
    f"gpu_memory_utilization={GPU_MEMORY_UTIL}, "
    f"batch_size={BATCH_SIZE}, "
    f"max_tokens={MAX_TOKENS}"
)


def build_llm_kwargs():
    kwargs = dict(
        model=MODEL_ID,
        dtype="bfloat16",
        quantization=None,
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        max_model_len=MAX_MODEL_LEN,
        tensor_parallel_size=1,
        trust_remote_code=False,
        enforce_eager=True,
        enable_prefix_caching=False,
        disable_log_stats=True,
        reasoning_parser="qwen3",
        limit_mm_per_prompt={"image": 0, "video": 0},

        # Critical Blackwell fix:
        # Force Triton MoE; do not allow auto-selection of FlashInfer CUTLASS MoE.
        kernel_config=KernelConfig(moe_backend="triton"),
    )

    sig = inspect.signature(LLM.__init__)
    if "language_model_only" in sig.parameters:
        kwargs["language_model_only"] = True

    return kwargs


def make_sampling_params(schema):
    base = dict(
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    if StructuredOutputsParams is not None:
        try:
            return SamplingParams(
                **base,
                structured_outputs=StructuredOutputsParams(json=schema),
            )
        except TypeError:
            pass

    if GuidedDecodingParams is not None:
        return SamplingParams(
            **base,
            guided_decoding=GuidedDecodingParams(json=schema),
        )

    return SamplingParams(**base)


llm = LLM(**build_llm_kwargs())
tokenizer = llm.get_tokenizer()
sampling_params = make_sampling_params(RAG_SCHEMA)

print("Model loaded successfully.")

Loading Qwen3.5 with vLLM offline inference...
MODEL_ID=Qwen/Qwen3.5-35B-A3B
max_model_len=4096, gpu_memory_utilization=0.9, batch_size=32, max_tokens=384
INFO 04-30 18:59:44 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'reasoning_parser': 'qwen3', 'kernel_config': KernelConfig(ir_op_priority=IrOpPriorityConfig(rms_norm=[]), enable_flashinfer_autotune=None, moe_backend='triton'), 'model': 'Qwen/Qwen3.5-35B-A3B'}


config.json: 0.00B [00:00, ?B/s]

WARNING 04-30 18:59:45 [arg_utils.py:1467] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

INFO 04-30 18:59:56 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 04-30 18:59:56 [nixl_utils.py:34] NIXL is not available
WARNING 04-30 18:59:56 [nixl_utils.py:44] NIXL agent config is not available
INFO 04-30 18:59:56 [model.py:555] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 04-30 18:59:56 [model.py:1680] Using max model len 4096
INFO 04-30 18:59:56 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-30 18:59:56 [vllm.py:840] Asynchronous scheduling is enabled.
WARNING 04-30 18:59:56 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-30 18:59:56 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-30 18:59:56 [kernel.py:205] Final IR op priority 

...

INFO 04-30 19:03:24 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3.5-35B-A3B: 201.004344 seconds
INFO 04-30 19:03:24 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 66.97 GiB. Available RAM: 169.14 GiB.
INFO 04-30 19:03:24 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]


INFO 04-30 19:03:32 [default_loader.py:384] Loading weights took 7.44 seconds
INFO 04-30 19:03:32 [unquantized.py:343] Using MoEPrepareAndFinalizeNoDPEPModular
INFO 04-30 19:03:32 [gpu_model_runner.py:4879] Model loading took 64.69 GiB memory and 210.156106 seconds
INFO 04-30 19:03:32 [interface.py:606] Setting attention block size to 1056 tokens to ensure that attention page size is >= mamba page size.
INFO 04-30 19:03:32 [interface.py:630] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
WARNING 04-30 19:04:11 [fused_moe.py:1091] Using default MoE config. Performance might be sub-optimal! Config file not found at /usr/local/lib/python3.12/dist-packages/vllm/model_executor/layers/fused_moe/configs/E=256,N=512,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition.json
INFO 04-30 19:04:17 [gpu_worker.py:440] Available KV cache memory: 16.03 GiB
INFO 04-30 19:04:17 [kv_cache_utils.py:1711] GPU KV cache size: 209,088 tokens
INFO

2026-04-30 19:04:17,655 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
2026-04-30 19:04:17,866 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends


INFO 04-30 19:04:18 [core.py:306] init engine (profile, create kv cache, warmup model) took 45.09 s
Model loaded successfully.


## 6 · Run enrichment

In [14]:
from tqdm.auto import tqdm
import json
import time

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input not found: {INPUT_FILE}")

start = 0
if CHECKPOINT_FILE.exists():
    try:
        start = int(CHECKPOINT_FILE.read_text().strip() or "0")
    except ValueError:
        start = 0

print(f"Resuming at line {start:,}")

total = count_lines(INPUT_FILE)
target_total = min(total, start + LIMIT) if LIMIT else total
print(f"Total={total:,}  To process={target_total - start:,}")
print(f"Using BATCH_SIZE={BATCH_SIZE}, MAX_TOKENS={MAX_TOKENS}")


def render_prompt(card: dict) -> str:
    """Render Qwen chat prompt. Disable thinking where the tokenizer template supports it."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(card)},
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        return prompt + "\nDo not output chain-of-thought. Output only the JSON object.\n"


def clean_model_json(raw: str) -> str:
    raw = (raw or "").strip()

    if raw.startswith("```json"):
        raw = raw[7:]
    if raw.startswith("```"):
        raw = raw[3:]
    if raw.endswith("```"):
        raw = raw[:-3]

    raw = raw.strip()

    if "</think>" in raw:
        raw = raw.split("</think>", 1)[1].strip()

    return raw


def normalize_enrichment(enriched: dict) -> dict:
    """Fill optional fields expected by downstream code and tag the method."""
    defaults = _stub("", "", [], [], role="reasoning", method="qwen35_35b_a3b_vllm")
    for key, value in defaults.items():
        enriched.setdefault(key, value)
    enriched["method"] = "qwen35_35b_a3b_vllm"
    return enriched


out_f = OUTPUT_FILE.open("a", encoding="utf-8")
pbar = tqdm(total=target_total, initial=start, desc="enrich", unit="card", smoothing=0.03)
pending = []
json_errors = 0
batch_count = 0
started_at = time.time()


def flush_batch():
    global pending, json_errors, batch_count

    if not pending:
        return

    batch_count += 1
    batch_size_now = len(pending)
    prompts = [render_prompt(card) for _, card in pending]

    t0 = time.time()

    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        use_tqdm=False,
    )

    dt = time.time() - t0
    cards_per_sec = batch_size_now / max(dt, 1e-9)

    for (line_idx, card), output in zip(pending, outputs):
        raw = clean_model_json(output.outputs[0].text)

        try:
            enriched = json.loads(raw)
            if not isinstance(enriched, dict):
                raise ValueError(f"Model output a {type(enriched).__name__} instead of a dict.")
            enriched = normalize_enrichment(enriched)

        except (json.JSONDecodeError, ValueError) as e:
            json_errors += 1
            enriched = _stub("", "", [], [], role="reasoning", method="json_parse_failed")
            enriched["raw_output"] = raw[:400]
            enriched["parse_error"] = str(e)[:200]

        card["rag_enrichment"] = enriched
        out_f.write(json.dumps(card, ensure_ascii=False) + "\n")

    out_f.flush()
    CHECKPOINT_FILE.write_text(str(pending[-1][0] + 1))
    pbar.update(len(pending))

    if batch_count == 1 or batch_count % 5 == 0:
        elapsed = time.time() - started_at
        done = pbar.n - start
        remaining = max(target_total - pbar.n, 0)
        avg_cps = done / max(elapsed, 1e-9)
        eta_min = remaining / max(avg_cps, 1e-9) / 60.0

        print(
            f"[batch {batch_count}] "
            f"size={batch_size_now}, "
            f"batch_time={dt:.1f}s, "
            f"batch_rate={cards_per_sec:.2f} cards/s, "
            f"avg_rate={avg_cps:.2f} cards/s, "
            f"eta≈{eta_min:.1f} min, "
            f"json_errors={json_errors}"
        )

    pending.clear()


# --- Main Loop ---
processed = 0

try:
    for line_idx, card in stream_input(INPUT_FILE, start):
        if LIMIT and processed >= LIMIT:
            break

        auto = auto_classify(card)
        if auto is not None:
            card["rag_enrichment"] = auto
            out_f.write(json.dumps(card, ensure_ascii=False) + "\n")
            CHECKPOINT_FILE.write_text(str(line_idx + 1))
            pbar.update(1)
            processed += 1
            continue

        pending.append((line_idx, card))

        if len(pending) >= BATCH_SIZE:
            flush_batch()

        processed += 1

    flush_batch()

finally:
    out_f.close()
    pbar.close()

print(f"Done. JSON parse errors: {json_errors}")
print(f"Output → {OUTPUT_FILE}")

Resuming at line 166
Total=2,476,315  To process=1,000
Using BATCH_SIZE=32, MAX_TOKENS=384


enrich:  14%|#4        | 166/1166 [00:00<?, ?card/s]

[batch 1] size=32, batch_time=77.1s, batch_rate=0.42 cards/s, avg_rate=0.41 cards/s, eta≈39.2 min, json_errors=29
[batch 5] size=32, batch_time=14.2s, batch_rate=2.25 cards/s, avg_rate=1.21 cards/s, eta≈11.5 min, json_errors=129
[batch 10] size=32, batch_time=14.1s, batch_rate=2.26 cards/s, avg_rate=1.62 cards/s, eta≈6.9 min, json_errors=269
[batch 15] size=32, batch_time=14.4s, batch_rate=2.23 cards/s, avg_rate=1.80 cards/s, eta≈4.6 min, json_errors=400
[batch 20] size=32, batch_time=14.0s, batch_rate=2.28 cards/s, avg_rate=1.93 cards/s, eta≈2.9 min, json_errors=535
[batch 25] size=32, batch_time=14.1s, batch_rate=2.27 cards/s, avg_rate=2.00 cards/s, eta≈1.4 min, json_errors=661
[batch 30] size=32, batch_time=14.1s, batch_rate=2.26 cards/s, avg_rate=2.05 cards/s, eta≈0.0 min, json_errors=800
Done. JSON parse errors: 801
Output → /content/drive/MyDrive/swiss_law/artifacts/court_authority_cards_rag.jsonl


## 7 · Verify output

In [15]:
from collections import Counter

roles    = Counter()
outcomes = Counter()
methods  = Counter()
total_out = 0
missing_required = 0

REQUIRED = RAG_SCHEMA['required']

with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        e    = card.get('rag_enrichment', {})
        total_out += 1
        roles[e.get('paragraph_role', 'MISSING')]    += 1
        outcomes[e.get('outcome_signal', 'MISSING')] += 1
        methods[e.get('method', 'MISSING')]          += 1
        if any(k not in e for k in REQUIRED):
            missing_required += 1

print(f'Total output cards : {total_out:,}')
print(f'Missing required   : {missing_required}')
print()
print('paragraph_role distribution:')
for k, v in roles.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')
print()
print('outcome_signal distribution:')
for k, v in outcomes.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')
print()
print('method distribution:')
for k, v in methods.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')

Total output cards : 1,166
Missing required   : 0

paragraph_role distribution:
  reasoning                    933  (80.0%)
  procedural                   144  (12.3%)
  disposition                   52  (4.5%)
  cost                          29  (2.5%)
  holding                        5  (0.4%)
  obiter                         1  (0.1%)
  standard_of_review             1  (0.1%)
  background                     1  (0.1%)

outcome_signal distribution:
  none                        1019  (87.4%)
  dismissed                     69  (5.9%)
  remitted                      56  (4.8%)
  inadmissible                  10  (0.9%)
  partial                        6  (0.5%)
  granted                        6  (0.5%)

method distribution:
  json_parse_failed            879  (75.4%)
  qwen35_35b_a3b_vllm          243  (20.8%)
  auto_short                    34  (2.9%)
  auto_cost                     10  (0.9%)


In [16]:
# Show 3 random LLM-enriched cards for a quick quality check
import random

llm_cards = []
with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        if card.get('rag_enrichment', {}).get('method', '').startswith('qwen35'):
            llm_cards.append(card)

for card in random.sample(llm_cards, min(3, len(llm_cards))):
    e = card['rag_enrichment']
    print('─' * 70)
    print('Citation     :', card.get('citation', ''))
    print('Role         :', e.get('paragraph_role'))
    print('Outcome      :', e.get('outcome_signal'))
    print('Topic        :', e.get('legal_topic'))
    print('Summary      :', e.get('english_summary', '')[:200])
    print('Keywords     :', e.get('search_keywords'))
    print('NL queries   :', e.get('natural_language_queries'))
    print()

──────────────────────────────────────────────────────────────────────
Citation     : BGE 145 IV 23 E. 3.4
Role         : disposition
Outcome      : granted
Topic        : criminal law and criminal procedure
Summary      : Public dissemination of written materials satisfies the objective elements of the offense.
Keywords     : ['public dissemination', 'written materials', 'objective elements', 'infraction', 'constituent elements']
NL queries   : ['What constitutes the objective elements of an offense involving public written dissemination?', 'Does public action through writing satisfy the objective requirements of an infraction?']

──────────────────────────────────────────────────────────────────────
Citation     : BGE 139 I 265 E. 3
Role         : disposition
Outcome      : dismissed
Topic        : Constitutional and Public Law
Summary      : The dispute and matter for adjudication is whether the Social Welfare Office of the City of St. Gallen correctly declined to enter upon the app